# 03 — Model selection, ablations and product interpretation

All results below come from the frozen H&M experiment. Model selection used validation; feature/candidate ablations on test are descriptive diagnostics.

In [ ]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPORT = ROOT / 'reports/hm_sample'
metrics = pd.read_csv(REPORT / 'metrics.csv')
json.loads((REPORT / 'selection.json').read_text())

In [ ]:
metrics[metrics['split'] == 'test'].sort_values('map@12', ascending=False)

In [ ]:
pd.read_csv(REPORT / 'tuning.csv')

## Temporal stability

Two expanding folds use fixed depth 5. The earlier fold and later fold can favor different models. These folds do not transfer later-selected parameters back into an earlier week.

In [ ]:
pd.read_csv(REPORT / 'rolling_validation.csv').pivot(index='model', columns='cutoff', values='map@12')

## Ablations

Candidate ablation fixes the ranker and retained-item features. Feature ablation refits the selected scorer on the same sampled training rows; source membership is still fixed. The two answer different questions.

In [ ]:
pd.read_csv(REPORT / 'ablations.csv')[['kind', 'removed', 'map@12', 'candidate_recall', 'delta_map']]

In [ ]:
segments = pd.read_csv(REPORT / 'segments.csv')
segments[(segments.model == 'xgboost_classifier') & segments.axis.isin(['activity', 'target_item_age'])]

In [ ]:
pd.read_csv(REPORT / 'catalog_diagnostics.csv').set_index('model')

In [ ]:
json.loads((REPORT / 'selected_vs_recent_bootstrap.json').read_text())

## Questions to answer in the write-up

1. Which source brings genuinely additional targets into the pool?
2. Why do recency and content similarity features help ordering, while ALS scores do not clearly help this sample?
3. How large is the loss for zero-history customers and future-unseen articles?
4. How does MAP improve while coverage declines versus recent purchases?
5. What must be logged to test conversion lift online?

The proposed experiment is in docs/online_ab_test.md. It has not been executed.